# ReFlex: spatial gRPC extraction (protocol v2)

Three 2D views show transmitted depth, the actual separable SOS-filter result, and geometric candidates over original depth. Missing samples are masked. ReFlex subsequently classifies absolute depth against its configured ZeroPlane, applies its MinDistance dead zone, normalises and tracks interactions.

Run in `observe` mode for live ReFlex data. Choose `demo` only while ReFlex processing is stopped: it sends two pushes and one pull through the real gRPC endpoint. Backend and framework must both support protocol v2.


In [ ]:
import asyncio
import json
import os
from pathlib import Path
import time

import grpc
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import clear_output, display
from generated_services import interactionService_pb2_grpc as rpc
from generated_services import interactionTypes_pb2 as messages

MODE = "observe"  # "demo" submits synthetic frames and changes filter settings.
DURATION_SECONDS = 60
REFRESH_SECONDS = 0.5
TARGET = os.getenv("GRPC_TARGET", "backend:50051")
SNAPSHOT = Path(os.getenv("SNAPSHOT_PATH", "/data/latest.json"))
assert MODE in {"observe", "demo"}
OPTIONS = [("grpc.max_receive_message_length", 8 * 1024 * 1024),
           ("grpc.max_send_message_length", 8 * 1024 * 1024)]

def call_rpc(method, request):
    with grpc.insecure_channel(TARGET, options=OPTIONS) as channel:
        return getattr(rpc.InteractionProcessingStub(channel), method)(request, timeout=3)

state = call_rpc("GetState", messages.StateRequest())
if state.protocol_version != 2:
    raise RuntimeError("Update backend and ReFlex together: protocol v2 required.")
print(f"Ready: {state.isReady}; frame: {state.frame_id}; mode: {MODE}")
if MODE == "demo":
    assert call_rpc("Configure", messages.ConfigRequest(cutoff=2, factor=100)).success

def send_demo_frame(elapsed):
    y, x = np.mgrid[:120, :160]
    shift = 3 * np.sin(elapsed)
    z = (1.45 - .12*np.exp(-((x-40-shift)**2+(y-35)**2)/180)
              - .10*np.exp(-((x-115)**2+(y-45-shift)**2)/180)
              + .11*np.exp(-((x-80)**2+(y-88)**2)/180))
    z[:4,:4] = np.nan
    prepared = time.perf_counter()
    request = messages.DepthValues1d(
        width=160, height=120, sample_step=1, zero_plane=1.45, z=z.ravel())
    preparation_ms = (time.perf_counter() - prepared) * 1000
    with grpc.insecure_channel(TARGET, options=OPTIONS) as channel:
        stub = rpc.InteractionProcessingStub(channel)
        started = time.perf_counter()
        response = stub.ComputeInteractions(request, timeout=3)
        roundtrip_ms = (time.perf_counter() - started) * 1000
        timings = {"client_request_construction": preparation_ms,
                   "client_rpc_roundtrip": roundtrip_ms}
        if "server_total" in response.timing.milliseconds:
            timings["client_transport_serialization_overhead"] = max(
                0, roundtrip_ms - response.timing.milliseconds["server_total"])
        stub.ReportTiming(messages.TimingSample(frame_id=response.frame_id,
                                               milliseconds=timings), timeout=3)
    return response


## Live spatial view

Run for 60 seconds, rerun to continue, or use **Kernel → Interrupt Kernel**. The plots use transmitted raster coordinates; multiply X/Y by `sample_step` for sensor coordinates. Frame age indicates whether the producer has stopped.

Candidate marker shape describes geometry: **v minimum**, **^ maximum**. Colour describes the depth comparison with ZeroPlane: red Push, blue Pull, gray equal. This comparison is illustrative: the notebook does not know ReFlex's configured dead zone and therefore does not display its final accepted interactions. The wire `type` is deliberately `NONE`.


In [ ]:
started = time.monotonic()
try:
    while time.monotonic() - started < DURATION_SECONDS:
        try:
            if MODE == "demo":
                send_demo_frame(time.monotonic() - started)
            snapshot = json.loads(SNAPSHOT.read_text(encoding="utf-8"))
            width, height = snapshot["width"], snapshot["height"]
            raw = np.asarray(snapshot["raw"], dtype=float).reshape(height, width)
            filtered = np.asarray(snapshot["filtered"], dtype=float).reshape(height, width)
            plane = snapshot["zero_plane"]
            age = max(0.0, time.time() - snapshot["received_at"])
            finite = raw[np.isfinite(raw)]
            amplitude = max(float(np.max(np.abs(finite-plane))) if finite.size else 0, .001)
            fig, axes = plt.subplots(1, 3, figsize=(15, 5), constrained_layout=True)
            titles = ["Input depth (missing samples masked)", "Separable Butterworth SOS", "Returned candidates (geometric extrema)"]
            for ax, values, title in zip(axes, [raw, filtered, raw], titles):
                # Preserve the original missing-data mask even after interpolation.
                masked = np.ma.array(values, mask=~np.isfinite(raw) | ~np.isfinite(values))
                im = ax.imshow(masked, origin="upper", cmap="coolwarm", vmin=plane-amplitude, vmax=plane+amplitude)
                ax.set_title(title)
                ax.set_xlabel("Raster X")
                ax.set_ylabel("Raster Y")
            for kind, marker in [("MINIMUM", "v"), ("MAXIMUM", "^")]:
                hits = [i for i in snapshot["interactions"] if i["extremum_kind"] == kind]
                colors = ["crimson" if i["z"] < plane else "royalblue" if i["z"] > plane else "gray" for i in hits]
                axes[2].scatter([i["point_idx"] % width for i in hits],
                                [i["point_idx"] // width for i in hits], c=colors,
                                marker=marker, edgecolors="white", linewidths=1.5, s=110, zorder=5, label=f"{kind}: {len(hits)}")
            axes[2].legend(loc="upper right")
            fig.colorbar(im, ax=axes, label="Absolute sensor depth", shrink=.7)
            status = "STALE / producer stopped?" if age > 3 else "receiving"
            fig.suptitle(f"Frame {snapshot['frame_id']} | age {age:.1f}s | {status} | step {snapshot['sample_step']}\n"
                         f"ZeroPlane={plane:.4f}; filter={snapshot['config']}; depth colours exclude ReFlex dead zone")
            clear_output(wait=True)
            display(fig)
            plt.close(fig)
            print(json.dumps(snapshot["interactions"][:10], indent=2))
            print("Backend stages (ms):", snapshot.get("timings_ms", "Update backend for timings"))
        except FileNotFoundError:
            clear_output(wait=True)
            print("Waiting for a protocol-v2 frame. Start ReFlex processing or choose demo mode.")
        except KeyError as error:
            raise RuntimeError("Snapshot uses the old schema. Rebuild/restart the backend and update ReFlex.") from error
        except grpc.RpcError as error:
            clear_output(wait=True)
            print(f"gRPC {error.code().name}: {error.details()}; retrying…")
        await asyncio.sleep(REFRESH_SECONDS)
except KeyboardInterrupt:
    print("Live view stopped.")
finally:
    plt.close("all")


## Interpretation and lifecycle

The processor fills missing samples with their nearest valid neighbour for filtering only, applies an order-5 SOS filter along both axes, detects spatial extrema and verifies contrast against original neighbouring depths. Returned depths remain original measurements. A saddle between deformations is not a 2D extremum. Additional supported extrema may exist; filter ringing alone is rejected.

`sample_step = NumSkipValues + 1` applies to both axes. Effective filter sampling rate is `factor / sample_step`; require `cutoff < factor / (2 * sample_step)`. Defaults are cutoff 2 and factor 100. Saved ReFlex settings are not migrated: set these values explicitly when upgrading. The service now applies configured values when connecting.

Snapshots are atomic, contain JSON `null` for missing values, and retain one frame. Snapshot JSON construction and atomic writes run in a separate process; only bounded enqueue work remains in the processing pipeline. Pending snapshots may be replaced by newer frames, so the spatial view can skip frames. Saved notebooks persist in the named volume, including across image rebuilds. Export edits before replacing an old notebook with this protocol-v2 version. IDs are per frame; timestamps retain the backend's Unix-millisecond convention.


## Pipeline timings: one-minute collection

Run the cells below independently of the spatial view. `await measure_pipeline()` collects **new successful frames for 60 seconds**, then returns raw samples and a statistical table (count, mean, standard deviation, minimum, median, p95, p99, maximum; all durations in ms), and draws distributions and a time series. Repeated reports for one frame are merged, so polling does not bias statistics towards slow frames. Missing client stages remain NaN, not zero. Raw data and statistics can be exported with `.to_csv(...)`.

For live ReFlex timings, enable **MeasurePerformance** in the tracking server and select the Remote processor. Observe mode only reads the bounded timing history; demo mode sends synthetic frames and measures the notebook client's request construction and RPC instead. Demo timings do not include C# raster conversion or ReFlex classification/smoothing. Stop the spatial loop before collecting.

`client_rpc_roundtrip` includes request serialization, transport, server processing, response serialization/deserialization and scheduling. `client_transport_serialization_overhead = max(0, roundtrip - server_total)` is an estimate of their combined overhead, **not pure network latency**. Monotonic clocks measure local durations; clock synchronisation is unnecessary. Request construction measures building protobuf objects, not the gRPC serializer itself.

`server_total` includes backend lock waiting and all synchronous server stages, including `server_snapshot_enqueue`. Snapshot construction and publication run in a separate process: `snapshot_background_write` reports their duration only for published snapshots and is outside the RPC total. `client_remote_total` covers the remote Analyze call, including preparation and response conversion; depth conversion, extremum classification/cleanup and smoothing follow it. Totals overlap component stages: do not sum every column. The timing-report RPC, notebook polling and plotting are excluded from reported stage durations, but add diagnostic load. Snapshot timings printed above describe processing before enqueue; completed background writes are reported separately in the history below. Background durations must not be added to synchronous pipeline latency. The pending queue retains at most one frame and replaces stale work, so not every frame becomes a snapshot.

The backend retains the last 4096 timing events (including updates to frames). Buffer overflow is reported; a backend restart aborts collection so frame IDs from different sessions cannot mix. Failed RPCs and skipped/busy frames are not latency samples. Without new frames the function returns empty tables and explains how to start the producer. Interrupting the kernel still evaluates already collected samples.


In [ ]:
async def measure_pipeline(duration_seconds=60, poll_seconds=0.25):
    """Collect every new successful frame, deduplicate reports, summarise and plot."""
    if not np.isfinite(duration_seconds) or duration_seconds <= 0:
        raise ValueError("duration_seconds must be finite and > 0")
    if not np.isfinite(poll_seconds) or poll_seconds <= 0:
        raise ValueError("poll_seconds must be finite and > 0")
    frames = {}
    lost_events = 0
    with grpc.insecure_channel(TARGET, options=OPTIONS) as channel:
        stub = rpc.InteractionProcessingStub(channel)
        baseline = await asyncio.to_thread(stub.GetTimings, messages.TimingRequest(), timeout=3)
        session = baseline.session_id
        cursor = baseline.latest_sequence
        baseline_frame = max((s.frame_id for s in baseline.samples), default=0)
        started = time.monotonic()
        deadline = started + duration_seconds

        async def poll(existing_only=False):
            nonlocal cursor, lost_events
            batch = await asyncio.to_thread(stub.GetTimings,
                                           messages.TimingRequest(after_sequence=cursor), timeout=3)
            if batch.session_id != session:
                raise RuntimeError("Backend restarted during collection; rerun for a consistent session.")
            lost_events += max(0, batch.oldest_sequence - cursor - 1)
            for sample in batch.samples:
                if sample.frame_id <= baseline_frame or (existing_only and sample.frame_id not in frames):
                    continue
                row = frames.setdefault(sample.frame_id, {
                    "frame_id": sample.frame_id, "recorded_at_unix_ms": sample.recorded_at_unix_ms})
                row.update(sample.milliseconds)
            cursor = batch.latest_sequence

        print(f"Collecting {MODE} timings for {duration_seconds:g} seconds…")
        try:
            while time.monotonic() < deadline:
                if MODE == "demo":
                    await asyncio.to_thread(send_demo_frame, time.monotonic() - started)
                await poll()
                await asyncio.sleep(min(poll_seconds, max(0, deadline - time.monotonic())))
            # Merge late client reports for frames already inside the measurement window.
            await poll(existing_only=True)
        except KeyboardInterrupt:
            print("Interrupted; evaluating collected frames.")

    if lost_events:
        print(f"WARNING: {lost_events} timing events were overwritten before polling; results may be incomplete.")
    raw = pd.DataFrame(list(frames.values()))
    if raw.empty:
        print("No new frames. Start ReFlex remote processing (and MeasurePerformance for client stages), or select demo mode.")
        return raw, pd.DataFrame()
    raw = raw.sort_values("frame_id").set_index("frame_id")
    stages = sorted(c for c in raw if c.startswith(("server_", "client_", "snapshot_")))
    values = raw[stages]
    summary = values.describe(percentiles=[.5, .95, .99]).T
    summary = summary.rename(columns={"50%": "p50", "95%": "p95", "99%": "p99"})
    summary = summary[["count", "mean", "std", "min", "p50", "p95", "p99", "max"]]
    print(f"{len(raw)} distinct frames; {lost_events} overwritten events. Durations in ms.")
    if not any(c.startswith("client_") for c in stages):
        print("No client reports: enable ReFlex MeasurePerformance and update the tracking server.")
    display(summary.round(3))
    fig, (distributions, timeline) = plt.subplots(1, 2, figsize=(17, max(6, len(stages) * .28)),
                                                constrained_layout=True)
    distributions.boxplot([values[c].dropna().values for c in stages], vert=False, labels=stages,
                          showfliers=False)
    distributions.set_xlabel("Duration (ms); outliers hidden in plot, retained in statistics")
    distributions.set_title("Stage distributions (totals overlap; snapshot_ runs in background)")
    seconds = (raw["recorded_at_unix_ms"] - raw["recorded_at_unix_ms"].min()) / 1000
    for stage in ("server_total", "client_rpc_roundtrip", "client_remote_total"):
        if stage in values:
            timeline.plot(seconds, values[stage], label=stage, linewidth=1)
    timeline.set(xlabel="Seconds since first collected frame", ylabel="Duration (ms)",
                 title="Pipeline latency over time")
    timeline.legend()
    display(fig)
    plt.close(fig)
    return raw, summary


In [ ]:
timing_samples, timing_statistics = await measure_pipeline()
